# 🛰️ Live Satellite Tracker Notebook: Overview
## **Real-Time Positions → Ground Tracks → Live Swaths → ROI Countdown**

This notebook gives an **n2yo-style live view** centred on your ROI: where each satellite is now, its ground track for the last 45 and next 90 minutes, its current instrument swath, the day/night terminator, and a countdown to each satellite's next geometric coverage opportunity, with clocks in **UTC** and the **ROI's local time**. Sentinel-1C/D use nominal right-looking IW geometry by default (select EW in `predict_overpasses`).

---

### 🚀 Two Ways to Watch
| Option | Runs where | Best for |
| :--- | :--- | :--- |
| **`live_tracker`** (ipyleaflet) | Inside Jupyter, on Esri World Imagery | Working sessions — updates every 2 s while the notebook stays usable. |
| **`export_live_tracker_html`** | Any browser, no Python | Sharing, GitHub Pages, a second screen. Orbits are propagated in the browser (satellite.js / SGP4). |

> ⚠️ **Note**: ipyleaflet widgets need JupyterLab ≥ 3 or Notebook ≥ 7 with widgets enabled (included in the `notebooks` extra). GitHub cannot render live widgets.

## ⚙️ Workspace Configuration, ROI & Orbital Elements
> ### **Same ROI and TLE cache as the prediction notebook**

In [ ]:
# Override EO_OVERPASS_WORKSPACE when caches/reports live outside this checkout.
import os
from pathlib import Path

from eo_overpass_predictor import (
    export_live_tracker_html,
    fetch_tles,
    live_tracker,
    load_roi,
    next_overpasses,
    predict_overpasses,
    resolve_roi_timezone,
    satellite_positions,
)

REPO_DIR = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "pyproject.toml").is_file() and (p / "examples" / "data").is_dir()
    ),
    None,
)
if REPO_DIR is None:
    raise RuntimeError("Launch Jupyter from the repository root or notebooks/ directory.")

WORKSPACE_DIR = Path(os.environ.get("EO_OVERPASS_WORKSPACE", REPO_DIR)).expanduser().resolve()

# Areas: port_phillip_bay_australia.geojson, madrid_spain.geojson, austin_usa.geojson
# Points: cabauw_netherlands.geojson, railroad_valley_usa.geojson
ROI_FILE = "port_phillip_bay_australia.geojson"
ROI_PATH = REPO_DIR / "examples" / "data" / ROI_FILE
ROI_GDF = load_roi(ROI_PATH)
ROI_TZ = resolve_roi_timezone(ROI_GDF)
TLES = fetch_tles(tle_dir=(WORKSPACE_DIR / "DATA") / "TLE")

# Upcoming overpasses feed the countdown column
PASSES = predict_overpasses(
    ROI_GDF, TLES, days=16, timezone_name=ROI_TZ, s1_mode="IW", verbose=False
)
display(next_overpasses(PASSES))

## 📍 Current Positions (Snapshot)
A one-off table of sub-satellite position, altitude, ground speed and direction — the same numbers the live panel refreshes.

In [ ]:
display(satellite_positions(TLES))

## 📡 Live Tracker (ipyleaflet)
> ### **Async Update Loop → Map Layers → Telemetry Panel**

---

### 📂 Map Layers
| Layer | Meaning |
| :--- | :--- |
| **Dot** | Current sub-satellite point. |
| **Dashed / solid line** | Ground track, last 45 min / next 90 min (`track_minutes`). |
| **Shaded strip** | Instrument footprint for ±2.5 min around now. When it covers the red ROI, the ROI is geometrically covered; Sentinel-1 strips are nominal SAR capability. |
| **Dark overlay** | Night side of the terminator. |

Updates run as an `asyncio` task on the kernel's event loop, so other cells keep working. Use the **Live** toggle, or run the stop cell.

<div class="alert alert-block alert-warning" style="background-color:#fff3cd; color:#664d03; border:1px solid #ffe69c; border-left:6px solid #f0ad4e; border-radius:4px; padding:10px 16px; margin:6px 0;">
<b>⚠️ CAUTION — Live map display</b>
<ul style="margin:6px 0 0 0;">
  <li>The map needs <code>ipywidgets</code> and <code>ipyleaflet</code> in the environment that <b>launched JupyterLab</b>, not only in the kernel. From the repository root, run <code>uv sync --extra notebooks</code> → <code>uv run jupyter lab</code>.</li>
  <li>If you see a model ID or <i>Error displaying widget</i> instead of a map, that is the cause — the standalone page below works regardless.</li>
  <li>Positions come from the loaded TLEs; they are as current as the last <code>fetch_tles()</code> call.</li>
</ul>
</div>

In [ ]:
TRACKER = live_tracker(TLES, ROI_GDF, passes=PASSES, refresh_s=2, track_minutes=(45, 90), zoom=3)
TRACKER

In [ ]:
# Stop live updates (restart with TRACKER.start())
TRACKER.stop()

## 🌐 Standalone Live-Tracker Page
> ### **Embedded TLEs + Predictions → Browser-Side SGP4 → Shareable HTML**

Writes `OUTPUT/live_tracker.html`, a single page that keeps tracking in real time in any browser. It embeds the current TLEs, the ROI, the predicted overpasses and a Natural Earth coastline, and loads only `satellite.js` from a CDN. Drag to pan, scroll to zoom, switch between **World** and **ROI region**, and click a satellite to hide or show it.

> 💡 **Tip**: The embedded TLEs age. The page footer shows TLE age per satellite; re-export after refreshing TLEs.

<div class="alert alert-block alert-warning" style="background-color:#fff3cd; color:#664d03; border:1px solid #ffe69c; border-left:6px solid #f0ad4e; border-radius:4px; padding:10px 16px; margin:6px 0;">
<b>⚠️ CAUTION — Standalone page ages</b>
<ul style="margin:6px 0 0 0;">
  <li>The page <b>embeds today's TLEs</b>: positions drift as they age (seconds within a week, noticeably after a few weeks). The footer turns amber after 7 days.</li>
  <li>The countdown list ends when the prediction window ends (16 days by default), after which it reports no remaining overpasses.</li>
  <li><b>Re-export after refreshing TLEs</b> to keep a shared copy current.</li>
</ul>
</div>

In [ ]:
html_path = export_live_tracker_html(
    TLES, ROI_GDF, (WORKSPACE_DIR / "OUTPUT") / "live_tracker.html", passes=PASSES
)